config

In [1]:

import sys
import os
import tifffile
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import SimpleITK as sitk
sys.path.append('/home/cyf/wbi/Virginia/code_for_paper/wholistic_registration/src/wholistic_registration')
current_dir = os.path.dirname(os.path.abspath("__file__"))
root_dir = os.path.dirname(current_dir)
sys.path.append(root_dir)

root_dir = Path('/home/cyf/wbi/Virginia/code_for_paper/data/simulation_v0420/repos')
from utils import simulation
# root_dir = Path('/home/cyf/wbi/Virginia/code/src/wholistic_registration/exp/simulation/repos')
zslices = [2, 10, 18]  # 选择 z slice
experiment_groups = ['art_R', 'amp', 'noise_level']  # 对应三组实验
results = {}  # 保存 MSE
results = {group: {} for group in experiment_groups}
repo_dirs = sorted([d for d in root_dir.iterdir() if d.is_dir() and d.name.startswith('rep')])
num_repos = len(repo_dirs)
print(f"Found {num_repos} repos: {[d.name for d in repo_dirs]}")

CuPy not available - falling back to NumPy (CPU only)
CuPy not available - import standardscipy
Found 10 repos: ['rep01', 'rep02', 'rep03', 'rep04', 'rep05', 'rep06', 'rep07', 'rep08', 'rep09', 'rep10']


ITK_Diffeomorphic_Demons

In [2]:
import numpy as np
import SimpleITK as sitk


def run_demons_same_size(
    fixed_np,
    moving_np,
    spacing=(1.0, 1.0, 1.0),   # (sy, sx, sz) in your numpy convention
    method="diffeomorphic",    # "diffeomorphic" or "fast_symmetric"
    n_iter=80,
    smooth_disp_sigma=1.5,
    smooth_update_sigma=0.5,
    intensity_diff_threshold=0.001,
    max_update_step=0.5,
    use_image_spacing=True,
):
    """
    Same-size Demons registration.

    Parameters
    ----------
    fixed_np : np.ndarray
        Shape (Y, X, Z)
    moving_np : np.ndarray
        Shape (Y, X, Z)
    spacing : tuple
        (sy, sx, sz)

    Returns
    -------
    disp_np_yxz : np.ndarray
        Shape (Y, X, Z, 3), channel order [y, x, z]
    warped_np_yxz : np.ndarray
        Shape (Y, X, Z)
    out_tx : sitk.Transform
        Final displacement field transform
    """
    fixed_np = np.asarray(fixed_np, dtype=np.float32)
    moving_np = np.asarray(moving_np, dtype=np.float32)

    assert fixed_np.shape == moving_np.shape, "fixed and moving must have the same shape"
    assert fixed_np.ndim == 3, f"Expected 3D input, got ndim={fixed_np.ndim}"

    sy, sx, sz = spacing

    # Your numpy is (Y, X, Z) -> convert to SITK array order (Z, Y, X)
    fixed = sitk.GetImageFromArray(np.transpose(fixed_np, (2, 0, 1)))
    moving = sitk.GetImageFromArray(np.transpose(moving_np, (2, 0, 1)))

    # SITK spacing order is (x, y, z)
    fixed.SetSpacing((sx, sy, sz))
    moving.SetSpacing((sx, sy, sz))

    if method == "diffeomorphic":
        R = sitk.DiffeomorphicDemonsRegistrationFilter()
    elif method == "fast_symmetric":
        R = sitk.FastSymmetricForcesDemonsRegistrationFilter()
    else:
        raise ValueError(f"Unknown method: {method}")

    R.SetNumberOfIterations(int(n_iter))
    R.SetUseImageSpacing(bool(use_image_spacing))
    R.SetIntensityDifferenceThreshold(float(intensity_diff_threshold))

    R.SetSmoothDisplacementField(True)
    R.SetStandardDeviations(float(smooth_disp_sigma))

    if smooth_update_sigma > 0:
        R.SetSmoothUpdateField(True)
        R.SetUpdateFieldStandardDeviations(float(smooth_update_sigma))
    else:
        R.SetSmoothUpdateField(False)

    if hasattr(R, "SetMaximumUpdateStepLength"):
        R.SetMaximumUpdateStepLength(float(max_update_step))

    # Demons returns a displacement field image
    disp_field = R.Execute(fixed, moving)

    # Wrap as transform
    out_tx = sitk.DisplacementFieldTransform(disp_field)

    # Re-generate displacement field on the fixed grid explicitly
    displacement_field = sitk.TransformToDisplacementField(
        out_tx,
        sitk.sitkVectorFloat64,
        fixed.GetSize(),
        fixed.GetOrigin(),
        fixed.GetSpacing(),
        fixed.GetDirection()
    )

    motion_corrected_image = sitk.Resample(
        moving,
        fixed,
        out_tx,
        sitk.sitkLinear,
        float(np.min(moving_np)),
        moving.GetPixelID()
    )

    # Robust extraction: split vector image into 3 scalar channels
    disp_x_zyx = sitk.GetArrayFromImage(sitk.VectorIndexSelectionCast(displacement_field, 0))
    disp_y_zyx = sitk.GetArrayFromImage(sitk.VectorIndexSelectionCast(displacement_field, 1))
    disp_z_zyx = sitk.GetArrayFromImage(sitk.VectorIndexSelectionCast(displacement_field, 2))

    # (Z, Y, X) -> (Y, X, Z)
    disp_x_yxz = np.transpose(disp_x_zyx, (1, 2, 0)).astype(np.float32)
    disp_y_yxz = np.transpose(disp_y_zyx, (1, 2, 0)).astype(np.float32)
    disp_z_yxz = np.transpose(disp_z_zyx, (1, 2, 0)).astype(np.float32)

    # SITK displacement channel order is [x, y, z]
    # convert to your convention [y, x, z]
    disp_np_yxz = np.stack([disp_y_yxz, disp_x_yxz, disp_z_yxz], axis=-1)

    warped_zyx = sitk.GetArrayFromImage(motion_corrected_image)
    warped_np_yxz = np.transpose(warped_zyx, (1, 2, 0)).astype(np.float32)

    return disp_np_yxz, warped_np_yxz, out_tx

In [ ]:
import os
import time
import numpy as np
import tifffile
import SimpleITK as sitk
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

TOTAL_CPUS = os.cpu_count() or 8
THREADS_PER_CASE = 4
MAX_WORKERS = 32

print(f"[Parallel Config] TOTAL_CPUS={TOTAL_CPUS}, THREADS_PER_CASE={THREADS_PER_CASE}, MAX_WORKERS={MAX_WORKERS}")

# -------------------------
# demons 参数
# -------------------------
demons_params = {
    "method": "diffeomorphic",
    "n_iter": 80,
    "smooth_disp_sigma": 1.5,
    "smooth_update_sigma": 0.8,
    "intensity_diff_threshold": 0.001,
    "max_update_step": 0.5,
    "use_image_spacing": True,
}

crop = 50
z_ratio_eval = 3.0769230769230766 * 8
demons_spacing = (1.0, 1.0, 3.0769230769230766 * 8)


def process_one_case(repo_dir_str, group, exp_level, zslices, demons_params, crop, z_ratio_eval, demons_spacing, threads_per_case):
    """
    Process one experiment case.
    Return:
        (group, exp_level, metrics_dict)
    """
    # 每个进程内部限制 SITK 线程数，避免过度超卖 CPU
    sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(int(threads_per_case))

    repo_dir = Path(repo_dir_str)
    subfolder = repo_dir / group / exp_level

    ref_path = subfolder / 'ref.tif'
    mov_path = subfolder / 'mov.tif'
    motion_path = subfolder / 'motion.tif'

    metrics = {
        'demons_intensity_mse': np.nan,
        'demons_intensity_rmse': np.nan,
        'demons_intensity_ssim': np.nan,
        'demons_runtime_sec': np.nan,
        'demons_motion_epe_xy_mean': np.nan,
        'demons_motion_epe_xy_median': np.nan,
        'demons_motion_rmse_xy': np.nan,
        'demons_motion_mae_y': np.nan,
        'demons_motion_mae_x': np.nan,
        'demons_motion_rmse_y': np.nan,
        'demons_motion_rmse_x': np.nan,
        'demons_motion_epe_3d_mean': np.nan,
        'demons_motion_epe_3d_median': np.nan,
        'demons_motion_epe_phys_mean': np.nan,
        'demons_motion_epe_phys_median': np.nan,
        'demons_motion_rmse_z': np.nan,
        'demons_motion_mae_z': np.nan,
    }

    try:
        if not ref_path.exists() or not mov_path.exists() or not motion_path.exists():
            return group, exp_level, metrics

        ref_data = tifffile.imread(ref_path)[..., 2:19]
        move_data = tifffile.imread(mov_path)[..., zslices]
        motion_data = tifffile.imread(motion_path)[:, :, zslices, :]

        ref_data_clip = ref_data[:, :, [0, 8, 16]]

        mov_eval = simulation.crop_valid(move_data, crop=crop)
        gt_motion_eval = simulation.crop_valid(motion_data, crop=crop)

        t0 = time.perf_counter()
        motion_demons, data_mov_demons, _ = run_demons_same_size(
            fixed_np=move_data,
            moving_np=ref_data_clip,
            spacing=demons_spacing,
            method=demons_params["method"],
            n_iter=demons_params["n_iter"],
            smooth_disp_sigma=demons_params["smooth_disp_sigma"],
            smooth_update_sigma=demons_params["smooth_update_sigma"],
            intensity_diff_threshold=demons_params["intensity_diff_threshold"],
            max_update_step=demons_params["max_update_step"],
            use_image_spacing=demons_params["use_image_spacing"],
        )
        demons_runtime_sec = time.perf_counter() - t0

        motion_demons_eval = simulation.crop_valid(motion_demons, crop=crop)
        demons_img_eval = simulation.crop_valid(data_mov_demons, crop=crop)
        gt_motion_eval = simulation.crop_valid(motion_data, crop=crop)
        
        metrics['demons_intensity_mse'] = simulation.compute_intensity_mse(demons_img_eval, mov_eval)
        metrics['demons_intensity_rmse'] = simulation.compute_intensity_rmse(demons_img_eval, mov_eval)
        metrics['demons_intensity_ssim'] = simulation.compute_volume_ssim(demons_img_eval, mov_eval)
        metrics['demons_runtime_sec'] = demons_runtime_sec

        demons_motion_metrics = simulation.compute_motion_metrics(
            motion_demons_eval,
            gt_motion_eval,
            z_ratio=z_ratio_eval,
            use_z=True
        )

        metrics['demons_motion_epe_xy_mean'] = demons_motion_metrics['motion_epe_xy_mean']
        metrics['demons_motion_epe_xy_median'] = demons_motion_metrics['motion_epe_xy_median']
        metrics['demons_motion_rmse_xy'] = demons_motion_metrics['motion_rmse_xy']
        metrics['demons_motion_mae_y'] = demons_motion_metrics['motion_mae_y']
        metrics['demons_motion_mae_x'] = demons_motion_metrics['motion_mae_x']
        metrics['demons_motion_rmse_y'] = demons_motion_metrics['motion_rmse_y']
        metrics['demons_motion_rmse_x'] = demons_motion_metrics['motion_rmse_x']

        if 'motion_epe_3d_mean' in demons_motion_metrics:
            metrics['demons_motion_epe_3d_mean'] = demons_motion_metrics['motion_epe_3d_mean']
            metrics['demons_motion_epe_3d_median'] = demons_motion_metrics['motion_epe_3d_median']
            metrics['demons_motion_epe_phys_mean'] = demons_motion_metrics['motion_epe_phys_mean']
            metrics['demons_motion_epe_phys_median'] = demons_motion_metrics['motion_epe_phys_median']
            metrics['demons_motion_rmse_z'] = demons_motion_metrics['motion_rmse_z']
            metrics['demons_motion_mae_z'] = demons_motion_metrics['motion_mae_z']

        print(
            f"[DONE] {repo_dir.name}/{group}/{exp_level} | "
            f"MSE={metrics['demons_intensity_mse']:.4f}, "
            f"EPE_xy={metrics['demons_motion_epe_xy_mean']:.4f}, "
            f"time={metrics['demons_runtime_sec']:.2f}s"
        )

    except Exception as e:
        print(f"[FAILED] {repo_dir.name}/{group}/{exp_level}: {e}")

    return group, exp_level, metrics


# -------------------------
# 收集所有任务
# -------------------------
tasks = []
for repo_dir in repo_dirs:
    for group in experiment_groups:
        group_dir = repo_dir / group
        if not group_dir.exists():
            continue
        for subfolder in sorted(group_dir.iterdir(), key=lambda x: int(x.name)):
            if not subfolder.is_dir():
                continue
            exp_level = subfolder.name
            tasks.append((str(repo_dir), group, exp_level))

print(f"[Task Count] {len(tasks)} cases")

# -------------------------
# 并行执行
# -------------------------
results = {group: {} for group in experiment_groups}

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(
            process_one_case,
            repo_dir_str,
            group,
            exp_level,
            zslices,
            demons_params,
            crop,
            z_ratio_eval,
            demons_spacing,
            THREADS_PER_CASE
        )
        for repo_dir_str, group, exp_level in tasks
    ]

    for future in as_completed(futures):
        group, exp_level, case_metrics = future.result()

        if exp_level not in results[group]:
            results[group][exp_level] = {
                'demons_intensity_mse': [],
                'demons_intensity_rmse': [],
                'demons_intensity_ssim': [],
                'demons_runtime_sec': [],
                'demons_motion_epe_xy_mean': [],
                'demons_motion_epe_xy_median': [],
                'demons_motion_rmse_xy': [],
                'demons_motion_mae_y': [],
                'demons_motion_mae_x': [],
                'demons_motion_rmse_y': [],
                'demons_motion_rmse_x': [],
                'demons_motion_epe_3d_mean': [],
                'demons_motion_epe_3d_median': [],
                'demons_motion_epe_phys_mean': [],
                'demons_motion_epe_phys_median': [],
                'demons_motion_rmse_z': [],
                'demons_motion_mae_z': [],
            }

        for k, v in case_metrics.items():
            results[group][exp_level][k].append(v)

# -------------------------
# processed_results 自动汇总
# -------------------------
processed_results = {}

for group in experiment_groups:
    if group not in results or not results[group]:
        continue

    sorted_exp_levels = sorted(results[group].keys(), key=int)

    processed_results[group] = {
        'labels': sorted_exp_levels
    }

    metric_names = set()
    for lv in sorted_exp_levels:
        metric_names.update(results[group][lv].keys())

    for metric_name in sorted(metric_names):
        processed_results[group][f'avg_{metric_name}'] = [
            float(np.nanmean(results[group][lv][metric_name])) if len(results[group][lv][metric_name]) > 0 else np.nan
            for lv in sorted_exp_levels
        ]
        processed_results[group][f'std_{metric_name}'] = [
            float(np.nanstd(results[group][lv][metric_name])) if len(results[group][lv][metric_name]) > 0 else np.nan
            for lv in sorted_exp_levels
        ]

# -------------------------
# 保存
# -------------------------
output_file = "/home/cyf/wbi/Virginia/code_for_paper/Fig2/2.1_simulation/result/Demons.npy"

np.save(output_file, {
    'raw_results': results,
    'processed_results': processed_results,
    'demons_params': demons_params,
    'parallel_config': {
        'total_cpus': TOTAL_CPUS,
        'threads_per_case': THREADS_PER_CASE,
        'max_workers': MAX_WORKERS,
    }
}, allow_pickle=True)

print(f"Saved all results to {output_file}")

[Parallel Config] TOTAL_CPUS=256, THREADS_PER_CASE=4, MAX_WORKERS=32
[Task Count] 290 cases
[DONE] rep01/art_R/10 | MSE=811.8346, EPE_xy=3.4943, time=65.51s
[DONE] rep01/amp/1 | MSE=934.2587, EPE_xy=2.7986, time=65.78s
[DONE] rep02/art_R/1 | MSE=1262.9438, EPE_xy=3.6281, time=65.31s
[DONE] rep01/art_R/9 | MSE=874.3735, EPE_xy=3.1592, time=65.39s
[DONE] rep01/noise_level/3 | MSE=1202.6351, EPE_xy=3.3722, time=65.98s
[DONE] rep02/art_R/2 | MSE=1269.5436, EPE_xy=3.7309, time=65.50s
[DONE] rep01/amp/10 | MSE=2298.0891, EPE_xy=7.3180, time=65.94s
[DONE] rep01/noise_level/6 | MSE=1715.5771, EPE_xy=3.7807, time=65.98s
[DONE] rep01/noise_level/2 | MSE=891.2639, EPE_xy=3.4411, time=66.10s
[DONE] rep01/amp/5 | MSE=1379.4520, EPE_xy=5.4115, time=66.55s
[DONE] rep01/art_R/1 | MSE=1036.2791, EPE_xy=3.6362, time=66.43s[DONE] rep01/art_R/8 | MSE=960.6069, EPE_xy=3.4760, time=66.81s

[DONE] rep01/art_R/7 | MSE=1521.9224, EPE_xy=3.0664, time=66.97s
[DONE] rep01/noise_level/4 | MSE=1132.3931, EPE_xy=3.0

In [18]:
import time
import SimpleITK as sitk
import tifffile
ref_path = '/home/cyf/wbi/Virginia/code_for_paper/data/simulation_v0420/repos/rep03/amp/1/ref.tif'
mov_path = '/home/cyf/wbi/Virginia/code_for_paper/data/simulation_v0420/repos/rep03/amp/1/mov.tif'
zslices=[2, 10, 18]  # 选择 z slice
ref_data = tifffile.imread(ref_path)[..., 2:19]
move_data = tifffile.imread(mov_path)[..., zslices]
ref_data_clip = ref_data[:, :, [0, 8, 16]]
for nt in [4]:
    sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(nt)

    t0 = time.perf_counter()
    motion_demons, data_mov_demons, _ = run_demons_same_size(
        fixed_np=move_data,
        moving_np=ref_data_clip,
        spacing=(1.0, 1.0, 3.0769230769230766*8),
        method="diffeomorphic",
        n_iter=80,
        smooth_disp_sigma=1.5,
        smooth_update_sigma=0.5,
        intensity_diff_threshold=0.001,
        max_update_step=0.5,
        use_image_spacing=True,
    )
    t1 = time.perf_counter()

    print(f"threads={nt}, runtime={t1-t0:.3f} s")


threads=4, runtime=50.542 s


In [19]:
import itertools
import numpy as np
motion_path = '/home/cyf/wbi/Virginia/code_for_paper/data/simulation_v0420/repos/rep03/amp/1/motion.tif'
motion_data = tifffile.imread(motion_path)[:, :, zslices, :]
crop = 50
pred = motion_demons[crop:-crop, crop:-crop, :, :]
gt   = motion_data[crop:-crop, crop:-crop, :, :]

perms = list(itertools.permutations([0, 1, 2]))

for perm in perms:
    pred_perm = pred[..., list(perm)]
    mse = np.mean((pred_perm - gt) ** 2)
    print(f"perm={perm}, mse={mse:.6f}, raw mse={np.mean(gt**2)}, motion amplitude = {np.mean(pred_perm**2)}")


perm=(0, 1, 2), mse=4.010054, raw mse=4.325291156768799, motion amplitude = 1.0823420286178589
perm=(0, 2, 1), mse=4.921801, raw mse=4.325291156768799, motion amplitude = 1.0823421478271484
perm=(1, 0, 2), mse=5.108061, raw mse=4.325291156768799, motion amplitude = 1.0823421478271484
perm=(1, 2, 0), mse=5.466486, raw mse=4.325291156768799, motion amplitude = 1.0823421478271484
perm=(2, 0, 1), mse=5.621559, raw mse=4.325291156768799, motion amplitude = 1.0823421478271484
perm=(2, 1, 0), mse=5.068235, raw mse=4.325291156768799, motion amplitude = 1.0823421478271484


In [ ]:

mov_eval = simulation.crop_valid(move_data, crop=crop)
gt_motion_eval = simulation.crop_valid(motion_data, crop=crop)
motion_demons_eval = simulation.crop_valid(motion_demons, crop=crop)
demons_img_eval = simulation.crop_valid(data_mov_demons, crop=crop)
gt_motion_eval = simulation.crop_valid(motion_data, crop=crop)

metrics = {}
metrics['demons_intensity_mse'] = simulation.compute_intensity_mse(demons_img_eval, mov_eval)
metrics['demons_intensity_rmse'] = simulation.compute_intensity_rmse(demons_img_eval, mov_eval)
metrics['demons_intensity_ssim'] = simulation.compute_volume_ssim(demons_img_eval, mov_eval)

demons_motion_metrics = simulation.compute_motion_metrics(
    motion_demons_eval,
    gt_motion_eval,
    z_ratio=z_ratio_eval,
    use_z=True
)

metrics['demons_motion_epe_xy_mean'] = demons_motion_metrics['motion_epe_xy_mean']
metrics['demons_motion_epe_xy_median'] = demons_motion_metrics['motion_epe_xy_median']
metrics['demons_motion_rmse_xy'] = demons_motion_metrics['motion_rmse_xy']
metrics['demons_motion_mae_y'] = demons_motion_metrics['motion_mae_y']
metrics['demons_motion_mae_x'] = demons_motion_metrics['motion_mae_x']
metrics['demons_motion_rmse_y'] = demons_motion_metrics['motion_rmse_y']
metrics['demons_motion_rmse_x'] = demons_motion_metrics['motion_rmse_x']

if 'motion_epe_3d_mean' in demons_motion_metrics:
    metrics['demons_motion_epe_3d_mean'] = demons_motion_metrics['motion_epe_3d_mean']
    metrics['demons_motion_epe_3d_median'] = demons_motion_metrics['motion_epe_3d_median']
    metrics['demons_motion_epe_phys_mean'] = demons_motion_metrics['motion_epe_phys_mean']
    metrics['demons_motion_epe_phys_median'] = demons_motion_metrics['motion_epe_phys_median']
    metrics['demons_motion_rmse_z'] = demons_motion_metrics['motion_rmse_z']
    metrics['demons_motion_mae_z'] = demons_motion_metrics['motion_mae_z']